# Feature Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_Mech_Interp/blob/main/lectures/02_early_cnn_mechinterp/exercises/01_feature_viz/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_Mech_Interp/main/lectures/02_early_cnn_mechinterp/exercises/01_feature_viz/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import get_model, get_activation, test_feature_viz_step, test_feature_viz

In [ ]:
import torch
import matplotlib.pyplot as plt
from einops import rearrange

model = get_model()

## Feature Visualization via Gradient Ascent

We want to understand what a CNN channel "looks for" by finding an input image that maximally activates it.

The idea: start from a random image and iteratively update it via gradient ascent to maximize the mean activation of a chosen channel.

We use a pretrained VGG19. Its `.features` attribute is a `nn.Sequential` of conv layers, ReLU activations, and max-pooling layers. `get_activation(model, layer_index, img)` runs the image through layers `0..layer_index` and returns the activation tensor.

### Exercise 1: Implement a single gradient ascent step

Given an image tensor `img` (shape `(1, 3, H, W)`, requires grad), compute the mean activation of the target channel, backpropagate, and return a new image stepped in the direction of the gradient.

Steps:
1. Call `img.retain_grad()` (needed because `img` becomes non-leaf after the first step)
2. Get the activation at `layer_index` using `get_activation`
3. Compute the mean of `activation[0, channel]` as the loss
4. Backpropagate
5. Update: `img_new = img + lr * img.grad / (img.grad.norm() + 1e-8)` (normalized gradient ascent)
6. Clamp to `[-2.5, 2.5]` to keep values in a reasonable range
7. Detach and set `requires_grad=True` for the next step

The gradient normalization in step 5 makes the step size independent of the gradient magnitude, which produces much better images.

In [ ]:
def feature_viz_step(img, model, layer_index, channel, lr=0.05):
    # TODO: implement gradient ascent step
    pass

In [ ]:
test_feature_viz_step(feature_viz_step)

<details>
<summary><b>Hint</b></summary>

The update must happen inside `torch.no_grad()` to avoid tracking the gradient step itself in the computation graph.

</details>

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def feature_viz_step(img, model, layer_index, channel, lr=0.05):
    img.retain_grad()
    act = get_activation(model, layer_index, img)
    loss = act[0, channel].mean()
    loss.backward()
    with torch.no_grad():
        img_new = img + lr * img.grad / (img.grad.norm() + 1e-8)
        img_new = img_new.clamp(-2.5, 2.5)
    return img_new.detach().requires_grad_(True)
```
</details>

### Exercise 2: Full visualization loop

Now put it together: start from a small random image and run `feature_viz_step` for many iterations.

1. Initialize: `img = 0.01 * torch.randn(1, 3, size, size, requires_grad=True)`
2. Run `feature_viz_step` for `steps` iterations
3. Return `img[0]` (shape `(3, H, W)`)

In [ ]:
def feature_viz(model, layer_index, channel, size=128, steps=256, lr=0.05):
    # TODO: implement the full loop
    pass

In [ ]:
test_feature_viz(feature_viz)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def feature_viz(model, layer_index, channel, size=128, steps=256, lr=0.05):
    img = 0.01 * torch.randn(1, 3, size, size, requires_grad=True)
    for _ in range(steps):
        img = feature_viz_step(img, model, layer_index, channel, lr)
    return img[0]
```
</details>

### Visualize!

Let's see what different channels at different layers look for. Early layers (layer 5) detect simple edges and textures. Deeper layers (layer 19) detect more complex patterns.

Run the cells below. Each one takes a minute or two.

In [ ]:
def show_features(model, layer_index, channels, size=128, steps=256):
    fig, axes = plt.subplots(1, len(channels), figsize=(4 * len(channels), 4))
    for ax, ch in zip(axes, channels):
        result = feature_viz(model, layer_index, ch, size=size, steps=steps)
        img = rearrange(result, 'c h w -> h w c').detach().numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img)
        ax.set_title(f'Layer {layer_index}, Ch {ch}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Early layer: edges and oriented stripes
show_features(model, layer_index=5, channels=[10, 20, 40, 60])

In [ ]:
# Deeper layer: more complex textures and patterns
show_features(model, layer_index=10, channels=[20, 30, 80, 100])

In [ ]:
# Even deeper: high-level texture features
show_features(model, layer_index=19, channels=[10, 30, 50, 100], size=224, steps=512)